# Temporal Splitting & Validation Design

In [2]:
# Load engineered dataset

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path(
    "../data/engineered/ronaldo_1000_ml_engineered.csv"
)

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"]
)

df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)
print("Unique match IDs:", df["match_id"].nunique())
print("Total goals:", df["target_goals"].sum())

print(
    "Date range:",
    df["date"].min().date(),
    "to",
    df["date"].max().date()
)

print(
    "Chronological:",
    df["date"].is_monotonic_increasing
)

print("\nTarget distribution:")
print(
    df["target_goals"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nLast 5 appearances:")
print(
    df[
        [
            "date",
            "team",
            "opponent",
            "competition",
            "target_goals",
        ]
    ]
    .tail()
    .to_string(index=False)
)

Shape: (1333, 100)
Unique match IDs: 1333
Total goals: 978
Date range: 2002-08-14 to 2026-08-28
Chronological: True

Target distribution:
target_goals
0    676
1    415
2    176
3     55
4      9
5      2

Last 5 appearances:
      date     team   opponent competition  target_goals
2026-07-02 Portugal    Croatia   World Cup             1
2026-07-06 Portugal      Spain   World Cup             0
2026-08-21 Al-Nassr  Al Riyadh  Pro League             1
2026-08-25 Al-Nassr Al-Ettifaq  Pro League             0
2026-08-28 Al-Nassr  Al Taawon  Pro League             1


## Inspect candidate temporal cutoffs

In [5]:
# Inspect candidate temporal validation cutoffs

candidate_cutoffs = [
    "2018-07-01",  # Juventus-era transition
    "2021-07-01",  # Manchester United return approaching
    "2023-01-01",  # Al-Nassr / late-career regime
    "2024-01-01",
    "2025-01-01",
    "2026-01-01",
]

rows = []

for cutoff in candidate_cutoffs:
    cutoff = pd.Timestamp(cutoff)

    before = df[df["date"] < cutoff]
    after = df[df["date"] >= cutoff]

    rows.append({
        "cutoff": cutoff.date(),
        "train_matches": len(before),
        "train_goals": before["target_goals"].sum(),
        "future_matches": len(after),
        "future_goals": after["target_goals"].sum(),
        "future_gpg": (
            after["target_goals"].mean()
            if len(after) > 0 else np.nan
        ),
        "future_scoring_rate": (
            after["target_scored"].mean()
            if len(after) > 0 else np.nan
        ),
    })

cutoff_summary = pd.DataFrame(rows)

print("CANDIDATE TEMPORAL CUTOFFS")
print("=" * 75)

print(
    cutoff_summary
    .round({
        "future_gpg": 3,
        "future_scoring_rate": 3,
    })
    .to_string(index=False)
)

print("\nCalendar-year match counts:")
print(
    df.groupby(df["date"].dt.year)
    .agg(
        matches=("match_id", "count"),
        goals=("target_goals", "sum"),
    )
    .tail(10)
    .to_string()
)

CANDIDATE TEMPORAL CUTOFFS
    cutoff  train_matches  train_goals  future_matches  future_goals  future_gpg  future_scoring_rate
2018-07-01            915          658             418           320       0.766                0.550
2021-07-01           1073          783             260           195       0.750                0.542
2023-01-01           1145          819             188           159       0.846                0.617
2024-01-01           1204          873             129           105       0.814                0.620
2025-01-01           1255          916              78            62       0.795                0.628
2026-01-01           1301          957              32            21       0.656                0.531

Calendar-year match counts:
      matches  goals
date                
2017       60     53
2018       53     49
2019       50     39
2020       45     44
2021       65     47
2022       46     16
2023       59     54
2024       51     43
2025       46     41

In [7]:
# Inspect team/regime composition by time period

periods = [
    ("Pre-2018/19", "2002-01-01", "2018-07-01"),
    ("2018/19–2020/21", "2018-07-01", "2021-07-01"),
    ("2021/22–2022", "2021-07-01", "2023-01-01"),
    ("2023", "2023-01-01", "2024-01-01"),
    ("2024", "2024-01-01", "2025-01-01"),
    ("2025", "2025-01-01", "2026-01-01"),
    ("2026", "2026-01-01", "2027-01-01"),
]

summary_rows = []

for label, start, end in periods:
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    part = df[
        (df["date"] >= start) &
        (df["date"] < end)
    ]

    teams = (
        part["team"]
        .value_counts()
        .to_dict()
    )

    summary_rows.append({
        "period": label,
        "start": part["date"].min().date() if len(part) else None,
        "end": part["date"].max().date() if len(part) else None,
        "matches": len(part),
        "goals": part["target_goals"].sum(),
        "gpg": part["target_goals"].mean(),
        "scoring_rate": part["target_scored"].mean(),
        "teams": teams,
    })

regime_summary = pd.DataFrame(summary_rows)

pd.set_option("display.max_colwidth", None)

print(
    regime_summary.round({
        "gpg": 3,
        "scoring_rate": 3,
    }).to_string(index=False)
)

         period      start        end  matches  goals   gpg  scoring_rate                                                                              teams
    Pre-2018/19 2002-08-14 2018-06-30      915    658 0.719         0.467 {'Real Madrid': 438, 'Manchester United': 292, 'Portugal': 154, 'Sporting CP': 31}
2018/19–2020/21 2018-08-18 2021-06-27      158    125 0.791         0.563                                                  {'Juventus': 133, 'Portugal': 25}
   2021/22–2022 2021-08-22 2022-12-10       72     36 0.500         0.347                           {'Manchester United': 54, 'Portugal': 17, 'Juventus': 1}
           2023 2023-01-22 2023-12-30       59     54 0.915         0.610                                                    {'Al-Nassr': 50, 'Portugal': 9}
           2024 2024-02-14 2024-12-06       51     43 0.843         0.608                                                   {'Al-Nassr': 39, 'Portugal': 12}
           2025 2025-01-09 2025-12-30       46     41 0.89

## Test proposed validation architecture

In [10]:
# Inspect proposed expanding-window validation design

validation_windows = [
    {
        "fold": "Fold 1",
        "train_end": "2018-06-30",
        "val_start": "2018-07-01",
        "val_end": "2021-06-30",
    },
    {
        "fold": "Fold 2",
        "train_end": "2021-06-30",
        "val_start": "2021-07-01",
        "val_end": "2022-12-31",
    },
    {
        "fold": "Fold 3",
        "train_end": "2022-12-31",
        "val_start": "2023-01-01",
        "val_end": "2023-12-31",
    },
    {
        "fold": "Fold 4",
        "train_end": "2023-12-31",
        "val_start": "2024-01-01",
        "val_end": "2024-12-31",
    },
]

rows = []

for spec in validation_windows:
    train_end = pd.Timestamp(spec["train_end"])
    val_start = pd.Timestamp(spec["val_start"])
    val_end = pd.Timestamp(spec["val_end"])

    train = df[df["date"] <= train_end]

    val = df[
        (df["date"] >= val_start) &
        (df["date"] <= val_end)
    ]

    rows.append({
        "fold": spec["fold"],
        "train_start": train["date"].min().date(),
        "train_end": train["date"].max().date(),
        "train_matches": len(train),
        "train_goals": train["target_goals"].sum(),
        "val_start": val["date"].min().date(),
        "val_end": val["date"].max().date(),
        "val_matches": len(val),
        "val_goals": val["target_goals"].sum(),
        "val_gpg": val["target_goals"].mean(),
        "val_scoring_rate": val["target_scored"].mean(),
    })

validation_summary = pd.DataFrame(rows)

# Proposed untouched final test period
final_test = df[df["date"] >= "2025-01-01"]

print("EXPANDING-WINDOW VALIDATION")
print("=" * 100)

print(
    validation_summary.round({
        "val_gpg": 3,
        "val_scoring_rate": 3,
    }).to_string(index=False)
)

print("\nFINAL UNTOUCHED TEST")
print("=" * 50)

print(
    "Period:",
    final_test["date"].min().date(),
    "to",
    final_test["date"].max().date()
)

print("Matches:", len(final_test))
print("Goals:", final_test["target_goals"].sum())
print("GPG:", round(final_test["target_goals"].mean(), 3))
print(
    "Scoring rate:",
    round(final_test["target_scored"].mean(), 3)
)

print("\nTest team composition:")
print(final_test["team"].value_counts().to_string())

EXPANDING-WINDOW VALIDATION
  fold train_start  train_end  train_matches  train_goals  val_start    val_end  val_matches  val_goals  val_gpg  val_scoring_rate
Fold 1  2002-08-14 2018-06-30            915          658 2018-08-18 2021-06-27          158        125    0.791             0.563
Fold 2  2002-08-14 2021-06-27           1073          783 2021-08-22 2022-12-10           72         36    0.500             0.347
Fold 3  2002-08-14 2022-12-10           1145          819 2023-01-22 2023-12-30           59         54    0.915             0.610
Fold 4  2002-08-14 2023-12-30           1204          873 2024-02-14 2024-12-06           51         43    0.843             0.608

FINAL UNTOUCHED TEST
Period: 2025-01-09 to 2026-08-28
Matches: 78
Goals: 62
GPG: 0.795
Scoring rate: 0.628

Test team composition:
team
Al-Nassr    62
Portugal    16


In [12]:
# Build temporal validation indices

VALIDATION_WINDOWS = [
    ("Fold 1", "2018-06-30", "2018-07-01", "2021-06-30"),
    ("Fold 2", "2021-06-30", "2021-07-01", "2022-12-31"),
    ("Fold 3", "2022-12-31", "2023-01-01", "2023-12-31"),
    ("Fold 4", "2023-12-31", "2024-01-01", "2024-12-31"),
]

temporal_folds = []

for fold_name, train_end, val_start, val_end in VALIDATION_WINDOWS:

    train_idx = df.index[
        df["date"] <= pd.Timestamp(train_end)
    ].to_numpy()

    val_idx = df.index[
        (df["date"] >= pd.Timestamp(val_start)) &
        (df["date"] <= pd.Timestamp(val_end))
    ].to_numpy()

    temporal_folds.append(
        {
            "name": fold_name,
            "train_idx": train_idx,
            "val_idx": val_idx,
        }
    )


# Final untouched test
final_train_idx = df.index[
    df["date"] < pd.Timestamp("2025-01-01")
].to_numpy()

final_test_idx = df.index[
    df["date"] >= pd.Timestamp("2025-01-01")
].to_numpy()


# Integrity checks
print("TEMPORAL FOLD INTEGRITY")
print("=" * 75)

for fold in temporal_folds:

    train = df.loc[fold["train_idx"]]
    val = df.loc[fold["val_idx"]]

    no_overlap = set(fold["train_idx"]).isdisjoint(
        set(fold["val_idx"])
    )

    strict_time_order = (
        train["date"].max() < val["date"].min()
    )

    print(
        f'{fold["name"]}: '
        f'train={len(train):4d} | '
        f'validation={len(val):3d} | '
        f'no_overlap={no_overlap} | '
        f'strict_time_order={strict_time_order}'
    )


final_train = df.loc[final_train_idx]
final_test = df.loc[final_test_idx]

print("\nFINAL TRAIN / TEST")
print("=" * 75)

print(
    "Train:",
    final_train["date"].min().date(),
    "→",
    final_train["date"].max().date(),
    "| matches:",
    len(final_train)
)

print(
    "Test: ",
    final_test["date"].min().date(),
    "→",
    final_test["date"].max().date(),
    "| matches:",
    len(final_test)
)

print(
    "\nFinal train/test overlap:",
    not set(final_train_idx).isdisjoint(
        set(final_test_idx)
    )
)

print(
    "Strict final time order:",
    final_train["date"].max()
    < final_test["date"].min()
)

print(
    "Final test used inside validation:",
    any(
        len(
            set(final_test_idx)
            .intersection(set(fold["val_idx"]))
        ) > 0
        for fold in temporal_folds
    )
)

TEMPORAL FOLD INTEGRITY
Fold 1: train= 915 | validation=158 | no_overlap=True | strict_time_order=True
Fold 2: train=1073 | validation= 72 | no_overlap=True | strict_time_order=True
Fold 3: train=1145 | validation= 59 | no_overlap=True | strict_time_order=True
Fold 4: train=1204 | validation= 51 | no_overlap=True | strict_time_order=True

FINAL TRAIN / TEST
Train: 2002-08-14 → 2024-12-06 | matches: 1255
Test:  2025-01-09 → 2026-08-28 | matches: 78

Final train/test overlap: False
Strict final time order: True
Final test used inside validation: False


In [14]:
# Verify out-of-fold validation coverage

all_val_idx = np.concatenate([
    fold["val_idx"]
    for fold in temporal_folds
])

unique_val_idx = np.unique(all_val_idx)

validation_df = df.loc[unique_val_idx].copy()

print("OUT-OF-FOLD VALIDATION COVERAGE")
print("=" * 70)

print("Total validation predictions:", len(all_val_idx))
print("Unique validation matches:", len(unique_val_idx))

print(
    "Validation rows duplicated across folds:",
    len(all_val_idx) - len(unique_val_idx)
)

print(
    "Validation period:",
    validation_df["date"].min().date(),
    "→",
    validation_df["date"].max().date()
)

print("\nValidation matches by fold:")
for fold in temporal_folds:
    print(
        f'{fold["name"]:<8}',
        len(fold["val_idx"])
    )

print("\nCombined validation target:")
print(
    "Goals:",
    validation_df["target_goals"].sum()
)

print(
    "GPG:",
    round(
        validation_df["target_goals"].mean(),
        3
    )
)

print(
    "Scoring rate:",
    round(
        validation_df["target_scored"].mean(),
        3
    )
)

print("\nValidation / final-test separation:")

print(
    "OOF validation overlaps final test:",
    not set(unique_val_idx).isdisjoint(
        set(final_test_idx)
    )
)

print(
    "Latest validation match:",
    validation_df["date"].max().date()
)

print(
    "Earliest final-test match:",
    final_test["date"].min().date()
)

OUT-OF-FOLD VALIDATION COVERAGE
Total validation predictions: 340
Unique validation matches: 340
Validation rows duplicated across folds: 0
Validation period: 2018-08-18 → 2024-12-06

Validation matches by fold:
Fold 1   158
Fold 2   72
Fold 3   59
Fold 4   51

Combined validation target:
Goals: 258
GPG: 0.759
Scoring rate: 0.532

Validation / final-test separation:
OOF validation overlaps final test: False
Latest validation match: 2024-12-06
Earliest final-test match: 2025-01-09


## Define count-model evaluation metrics

In [17]:
# Define regression/count evaluation metrics

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance,
)

def evaluate_goal_predictions(y_true, y_pred):
    """
    Evaluate expected-goals predictions for Ronaldo's match goal count.

    y_pred represents the predicted expected number of goals.
    """

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # Poisson deviance requires strictly positive predictions.
    y_pred_safe = np.clip(y_pred, 1e-9, None)

    mae = mean_absolute_error(
        y_true,
        y_pred_safe
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred_safe
        )
    )

    poisson_deviance = mean_poisson_deviance(
        y_true,
        y_pred_safe
    )

    mean_prediction = y_pred_safe.mean()
    actual_mean = y_true.mean()

    mean_bias = mean_prediction - actual_mean

    return {
        "poisson_deviance": poisson_deviance,
        "mae": mae,
        "rmse": rmse,
        "actual_mean_goals": actual_mean,
        "predicted_mean_goals": mean_prediction,
        "mean_bias": mean_bias,
    }


# Sanity check using a constant prediction equal to
# the validation-set historical mean.
y_val_check = validation_df["target_goals"].to_numpy()

constant_prediction = np.full(
    len(y_val_check),
    y_val_check.mean()
)

metric_check = evaluate_goal_predictions(
    y_val_check,
    constant_prediction
)

print("METRIC FUNCTION SANITY CHECK")
print("=" * 50)

for metric, value in metric_check.items():
    print(f"{metric:<25} {value:.6f}")

METRIC FUNCTION SANITY CHECK
poisson_deviance          1.103082
mae                       0.709723
rmse                      0.864706
actual_mean_goals         0.758824
predicted_mean_goals      0.758824
mean_bias                 0.000000


## Expanding historical-mean baseline

In [20]:
# Step 4H — Evaluate expanding historical-mean baseline

baseline_fold_results = []
baseline_oof_predictions = []

for fold in temporal_folds:
    train = df.loc[fold["train_idx"]]
    val = df.loc[fold["val_idx"]]

    # Training information only
    train_goal_rate = train["target_goals"].mean()

    val_pred = np.full(
        len(val),
        train_goal_rate
    )

    metrics = evaluate_goal_predictions(
        val["target_goals"].to_numpy(),
        val_pred
    )

    baseline_fold_results.append({
        "fold": fold["name"],
        "train_matches": len(train),
        "val_matches": len(val),
        "train_goal_rate": train_goal_rate,
        **metrics,
    })

    baseline_oof_predictions.append(
        pd.DataFrame({
            "index": val.index,
            "y_true": val["target_goals"].to_numpy(),
            "y_pred": val_pred,
        })
    )


baseline_fold_results = pd.DataFrame(
    baseline_fold_results
)

baseline_oof = (
    pd.concat(
        baseline_oof_predictions,
        ignore_index=True
    )
    .sort_values("index")
)


# Macro metrics: each career regime receives equal weight
macro_metrics = (
    baseline_fold_results[
        [
            "poisson_deviance",
            "mae",
            "rmse",
            "mean_bias",
        ]
    ]
    .mean()
)


# Pooled metrics: all 340 OOF matches evaluated together
pooled_metrics = evaluate_goal_predictions(
    baseline_oof["y_true"],
    baseline_oof["y_pred"]
)


print("HISTORICAL-MEAN BASELINE — BY FOLD")
print("=" * 110)

print(
    baseline_fold_results[
        [
            "fold",
            "train_matches",
            "val_matches",
            "train_goal_rate",
            "actual_mean_goals",
            "predicted_mean_goals",
            "mean_bias",
            "poisson_deviance",
            "mae",
            "rmse",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


print("\nMACRO VALIDATION METRICS")
print("=" * 50)

for metric, value in macro_metrics.items():
    print(f"{metric:<25} {value:.6f}")


print("\nPOOLED OOF VALIDATION METRICS")
print("=" * 50)

for metric, value in pooled_metrics.items():
    print(f"{metric:<25} {value:.6f}")

HISTORICAL-MEAN BASELINE — BY FOLD
  fold  train_matches  val_matches  train_goal_rate  actual_mean_goals  predicted_mean_goals  mean_bias  poisson_deviance    mae   rmse
Fold 1            915          158           0.7191             0.7911                0.7191    -0.0720            1.0601 0.7001 0.8529
Fold 2           1073           72           0.7297             0.5000                0.7297     0.2297            1.2417 0.7230 0.8317
Fold 3           1145           59           0.7153             0.9153                0.7153    -0.2000            1.1418 0.7576 0.9472
Fold 4           1204           51           0.7251             0.8431                0.7251    -0.1181            1.0199 0.6867 0.8570

MACRO VALIDATION METRICS
poisson_deviance          1.115870
mae                       0.716870
rmse                      0.872184
mean_bias                 -0.040077

POOLED OOF VALIDATION METRICS
poisson_deviance          1.106694
mae                       0.712932
rmse             

In [22]:
# Evaluate 20-match recent-form baseline

recent_fold_results = []
recent_oof_predictions = []

for fold in temporal_folds:
    train = df.loc[fold["train_idx"]]
    val = df.loc[fold["val_idx"]]

    # Already reconstructed as a pre-match feature.
    val_pred = (
        val["goals_per_game_last_20"]
        .to_numpy(dtype=float)
    )

    metrics = evaluate_goal_predictions(
        val["target_goals"].to_numpy(),
        val_pred
    )

    recent_fold_results.append({
        "fold": fold["name"],
        "train_matches": len(train),
        "val_matches": len(val),
        **metrics,
    })

    recent_oof_predictions.append(
        pd.DataFrame({
            "index": val.index,
            "y_true": val["target_goals"].to_numpy(),
            "y_pred": val_pred,
        })
    )


recent_fold_results = pd.DataFrame(
    recent_fold_results
)

recent_oof = (
    pd.concat(
        recent_oof_predictions,
        ignore_index=True
    )
    .sort_values("index")
)


recent_macro_metrics = (
    recent_fold_results[
        [
            "poisson_deviance",
            "mae",
            "rmse",
            "mean_bias",
        ]
    ]
    .mean()
)


recent_pooled_metrics = evaluate_goal_predictions(
    recent_oof["y_true"],
    recent_oof["y_pred"]
)


print("20-MATCH RECENT-FORM BASELINE — BY FOLD")
print("=" * 100)

print(
    recent_fold_results[
        [
            "fold",
            "val_matches",
            "actual_mean_goals",
            "predicted_mean_goals",
            "mean_bias",
            "poisson_deviance",
            "mae",
            "rmse",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


print("\nMACRO VALIDATION METRICS")
print("=" * 50)

for metric, value in recent_macro_metrics.items():
    print(f"{metric:<25} {value:.6f}")


print("\nPOOLED OOF VALIDATION METRICS")
print("=" * 50)

for metric, value in recent_pooled_metrics.items():
    print(f"{metric:<25} {value:.6f}")


print("\nCOMPARISON WITH HISTORICAL-MEAN BASELINE")
print("=" * 65)

comparison = pd.DataFrame({
    "historical_mean": {
        "poisson_deviance": pooled_metrics["poisson_deviance"],
        "mae": pooled_metrics["mae"],
        "rmse": pooled_metrics["rmse"],
    },
    "recent_20": {
        "poisson_deviance": recent_pooled_metrics["poisson_deviance"],
        "mae": recent_pooled_metrics["mae"],
        "rmse": recent_pooled_metrics["rmse"],
    },
})

comparison["improvement_pct"] = (
    (
        comparison["historical_mean"]
        - comparison["recent_20"]
    )
    / comparison["historical_mean"]
    * 100
)

print(comparison.round(4).to_string())

20-MATCH RECENT-FORM BASELINE — BY FOLD
  fold  val_matches  actual_mean_goals  predicted_mean_goals  mean_bias  poisson_deviance    mae   rmse
Fold 1          158             0.7911                0.7934     0.0022            1.0979 0.6978 0.8711
Fold 2           72             0.5000                0.5861     0.0861            1.1694 0.6431 0.8059
Fold 3           59             0.9153                0.7839    -0.1314            1.3188 0.7398 0.9919
Fold 4           51             0.8431                0.8402    -0.0029            1.0325 0.6892 0.8655

MACRO VALIDATION METRICS
poisson_deviance          1.154657
mae                       0.692472
rmse                      0.883609
mean_bias                 -0.011493

POOLED OOF VALIDATION METRICS
poisson_deviance          1.141576
mae                       0.692206
rmse                      0.879359
actual_mean_goals         0.758824
predicted_mean_goals      0.754853
mean_bias                 -0.003971

COMPARISON WITH HISTORICAL-MEA

In [24]:
# Step 4J — Freeze temporal validation and model-selection policy

VALIDATION_POLICY = {
    "primary_target": "target_goals",

    # Primary metric for selecting count models
    "primary_metric": "poisson_deviance",

    # Supporting count metrics
    "secondary_metrics": [
        "mae",
        "rmse",
        "mean_bias",
    ],

    # How validation performance will be summarized
    "aggregation": [
        "macro_fold_average",
        "pooled_oof",
    ],

    # Final test is completely isolated from model selection
    "final_test_start": "2025-01-01",

    # Probability outputs evaluated later
    "probability_targets": [
        "target_scored",
        "target_multi_goal",
    ],

    # Selection principle
    "selection_rule": (
        "Prefer the model with the lowest macro Poisson deviance, "
        "while checking pooled Poisson deviance, MAE, RMSE, bias, "
        "fold stability, and probability calibration."
    ),
}


print("MODEL-SELECTION POLICY")
print("=" * 70)

for key, value in VALIDATION_POLICY.items():
    print(f"{key:<25}: {value}")


print("\nBASELINE REFERENCE")
print("=" * 70)

baseline_reference = pd.DataFrame({
    "historical_mean": {
        "macro_poisson_deviance":
            macro_metrics["poisson_deviance"],

        "pooled_poisson_deviance":
            pooled_metrics["poisson_deviance"],

        "pooled_mae":
            pooled_metrics["mae"],

        "pooled_rmse":
            pooled_metrics["rmse"],
    },

    "recent_20": {
        "macro_poisson_deviance":
            recent_macro_metrics["poisson_deviance"],

        "pooled_poisson_deviance":
            recent_pooled_metrics["poisson_deviance"],

        "pooled_mae":
            recent_pooled_metrics["mae"],

        "pooled_rmse":
            recent_pooled_metrics["rmse"],
    },
})

print(
    baseline_reference
    .round(6)
    .to_string()
)


# Final integrity assertions
assert len(temporal_folds) == 4
assert len(unique_val_idx) == 340
assert len(final_test_idx) == 78

assert set(unique_val_idx).isdisjoint(
    set(final_test_idx)
)

assert (
    validation_df["date"].max()
    < final_test["date"].min()
)

print("\nValidation policy integrity: PASSED")

MODEL-SELECTION POLICY
primary_target           : target_goals
primary_metric           : poisson_deviance
secondary_metrics        : ['mae', 'rmse', 'mean_bias']
aggregation              : ['macro_fold_average', 'pooled_oof']
final_test_start         : 2025-01-01
probability_targets      : ['target_scored', 'target_multi_goal']
selection_rule           : Prefer the model with the lowest macro Poisson deviance, while checking pooled Poisson deviance, MAE, RMSE, bias, fold stability, and probability calibration.

BASELINE REFERENCE
                         historical_mean  recent_20
macro_poisson_deviance          1.115870   1.154657
pooled_poisson_deviance         1.106694   1.141576
pooled_mae                      0.712932   0.692206
pooled_rmse                     0.866226   0.879359

Validation policy integrity: PASSED


In [26]:
# Step 4K — Save temporal folds and validation policy

import json
from pathlib import Path

ARTIFACT_DIR = Path("../artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


# Save fold boundaries rather than raw row indices.
# Date boundaries are more robust if the dataset is reloaded or re-indexed.
temporal_folds_spec = []

for fold in temporal_folds:
    train = df.loc[fold["train_idx"]]
    val = df.loc[fold["val_idx"]]

    temporal_folds_spec.append({
        "name": fold["name"],
        "train_start": str(train["date"].min().date()),
        "train_end": str(train["date"].max().date()),
        "validation_start": str(val["date"].min().date()),
        "validation_end": str(val["date"].max().date()),
        "train_matches": len(train),
        "validation_matches": len(val),
    })


TEMPORAL_FOLDS_PATH = (
    ARTIFACT_DIR / "temporal_folds.json"
)

with open(
    TEMPORAL_FOLDS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        temporal_folds_spec,
        f,
        indent=2
    )


VALIDATION_POLICY_PATH = (
    ARTIFACT_DIR / "validation_policy.json"
)

with open(
    VALIDATION_POLICY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        VALIDATION_POLICY,
        f,
        indent=2
    )


print("Saved:", TEMPORAL_FOLDS_PATH)
print("Saved:", VALIDATION_POLICY_PATH)

print("\nTemporal folds:")
for fold in temporal_folds_spec:
    print(
        f"{fold['name']}: "
        f"train {fold['train_start']} → {fold['train_end']} "
        f"({fold['train_matches']} matches) | "
        f"validation {fold['validation_start']} → "
        f"{fold['validation_end']} "
        f"({fold['validation_matches']} matches)"
    )

print("\nFinal test starts:")
print(
    VALIDATION_POLICY["final_test_start"]
)

Saved: ..\artifacts\temporal_folds.json
Saved: ..\artifacts\validation_policy.json

Temporal folds:
Fold 1: train 2002-08-14 → 2018-06-30 (915 matches) | validation 2018-08-18 → 2021-06-27 (158 matches)
Fold 2: train 2002-08-14 → 2021-06-27 (1073 matches) | validation 2021-08-22 → 2022-12-10 (72 matches)
Fold 3: train 2002-08-14 → 2022-12-10 (1145 matches) | validation 2023-01-22 → 2023-12-30 (59 matches)
Fold 4: train 2002-08-14 → 2023-12-30 (1204 matches) | validation 2024-02-14 → 2024-12-06 (51 matches)

Final test starts:
2025-01-01
